In [15]:
import json
import statistics
from pathlib import Path

In [16]:
# initialise directories
INPUT_DIR = Path('./Data/raw')
OUTPUT_DIR = Path('./Data/identified_videos')

In [17]:
def extract_outliers(file_path):
    """
    Identifies posts with significant play counts relative to average play counts
    of the creator's previous 30 posts.

    Parameters:
    - file_path: directory of the subfolder containing post data

    Returns:
    - outliers: video_id and play_count values of identified posts
    - median: the median play_count for the creator
    - mad: the median absolute deviation for the creator
    - threshold: the minimum play_count value required to be considered an outlier
    """

    # load data
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    videos = data['data']['itemList']

    video_data = []

    # get id and playCount values from raw data
    for video in videos:
        video_id = video.get('id')
        play_count = video.get('stats', {}).get('playCount')

        if video_id is not None and play_count is not None:
            video_data.append((video_id, int(play_count)))

    # handle exceptions
    if not video_data:
        return [], None, None, None

    play_counts = [count for _, count in video_data]

    # identify median play_count
    median = statistics.median(play_counts)

    # calculate absolute deviations
    absolute_deviations = [
        abs(x - median)
        for x in play_counts
    ]

    # calculate median of absolute deviations
    mad = statistics.median(absolute_deviations)

    # calculate threshold
    threshold = median + (2 * 1.4826 * mad)

    # get outliers
    outliers = [
        (video_id, play_count)
        for video_id, play_count in video_data
        if play_count > threshold
    ]

    return outliers, median, mad, threshold

In [18]:
def process_json_file(json_path: Path):
    try:
        # load JSON
        with json_path.open("r", encoding="utf-8") as f:
            data = json.load(f)

        # get comments
        comments = data.get("comments", [])

        # match output path while preserving subfolder and filename
        relative_path = json_path.relative_to(RAW_DIR)
        output_path = TAGGED_DIR / relative_path.with_suffix(".csv")

        # create output directory if none exists
        output_path.parent.mkdir(parents=True, exist_ok=True)

        # write to csv
        with output_path.open("w", encoding="utf-8", newline="") as f:
            writer = csv.writer(f)
            writer.writerow(["comment", "tag"])

            for comment in comments:
                text = comment.get("text", "")
                writer.writerow([text, ""])

        print(f"Processed: {json_path} -> {output_path}")
        
    # handle exceptions
    except Exception as e:
        print(f"Error processing {json_path.name}: {e}")

In [19]:
if __name__ == '__main__':
    main()

Results successfully saved to Data\identified_videos\ausunions_outliers.csv
Results successfully saved to Data\identified_videos\onenationoz_outliers.csv
Results successfully saved to Data\identified_videos\paulinehansononenation_outliers.csv
Results successfully saved to Data\identified_videos\weareunion_outliers.csv
